# **Block 1 — Imports**

This first block just sets up the notebook. We import a few standard libraries, then pull in the exact scikit-learn tools we need for the logistic regression workflow: loading a dataset, splitting train vs test, scaling features safely, training logistic regression, and evaluating predictions with a confusion matrix and metrics.

In [1]:

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
# Train/test split so we can evaluate generalization on unseen data
from sklearn.model_selection import train_test_split
# Pipeline lets us "chain" steps (scaling -> model) safely without data leakage
from sklearn.pipeline import Pipeline
# Scaling matters for logistic regression because features with huge scales can dominate training
from sklearn.preprocessing import StandardScaler
# Logistic Regression model for binary classification (predicting class probabilities)
from sklearn.linear_model import LogisticRegression
# Evaluation: confusion matrix + key classification metrics
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# **Block 2 — Load Dataset**

In supervised learning, we always separate inputs (features) from the output we want to predict (target). This block loads a clean binary dataset and converts it into a familiar pandas format so it’s easy to inspect and work with.

In [2]:
# Load the dataset object
data = load_breast_cancer()

# X = features (inputs the model uses to make a prediction)
X = pd.DataFrame(data.data, columns=data.feature_names)

# y = target/label (the correct answer for each row)
# For this dataset, y is binary (0 or 1), which is perfect for logistic regression
y = pd.Series(data.target, name="target")

# Quick sanity check: confirm we have rows and columns
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (569, 30)
y shape: (569,)


# **Block 3 — Quick Data Peek (What are we predicting?)**

Before training anything, we do a quick check to confirm the data looks reasonable and to see whether one class is much more common than the other. This matters because class imbalance can make “accuracy” look good even when the model is missing the cases we care about.

In [3]:
# View a few feature rows (inputs)
print("First 3 rows of X:")
display(X.head(3))

# Check class balance (how many 0s vs 1s)
# If one class dominates, accuracy alone can be misleading.
print("\nClass balance (counts):")
print(y.value_counts())

print("\nClass balance (percent):")
print((y.value_counts(normalize=True) * 100).round(2))


First 3 rows of X:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758



Class balance (counts):
target
1    357
0    212
Name: count, dtype: int64

Class balance (percent):
target
1    62.74
0    37.26
Name: proportion, dtype: float64


# **Block 4 — Train/Test Split (Honest evaluation)**

This block creates a training set for learning and a test set for evaluating. We use stratify=y so the train and test sets keep a similar class balance, and random_state so the split is reproducible.

In [4]:
# Split into train and test so we can measure generalization
# test_size=0.25 means 25% of the data is held out for testing
# stratify=y keeps the class proportions similar in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Confirm split sizes
print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes: ", X_test.shape, y_test.shape)

# Double-check that class balance is similar after stratification
print("\nTrain class %:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest class %:")
print((y_test.value_counts(normalize=True) * 100).round(2))


Train shapes: (426, 30) (426,)
Test shapes:  (143, 30) (143,)

Train class %:
target
1    62.68
0    37.32
Name: proportion, dtype: float64

Test class %:
target
1    62.94
0    37.06
Name: proportion, dtype: float64


# **Block 5 — Feature scaling (simple + correct)**

This block standardizes features because logistic regression is sensitive to feature scale. The key rule is: fit the scaler on training data only, then reuse it to transform both train and test, so you don’t leak test information.

In [5]:
# Create a scaler
scaler = StandardScaler()

# Fit ONLY on training data, then transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the SAME scaler (do NOT fit again)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)


Scaled train shape: (426, 30)
Scaled test shape : (143, 30)


# **Block 6 — Train logistic regression**

This block fits the model on the training set. We increase max_iter only to avoid convergence warnings; it does not change the core idea.

In [6]:
# Create the model
model = LogisticRegression(max_iter=1000)

# Train (fit) the model on the training data
model.fit(X_train_scaled, y_train)

print("Model trained.")


Model trained.


# **Block 7 — Get probabilities (the key output)**

This block is the “logistic regression moment.” The model does not start by giving a yes/no answer; it gives a probability for each class, and we usually focus on the probability of class 1.

In [7]:
# predict_proba returns probabilities for both classes:
# column 0 = P(class 0)
# column 1 = P(class 1)
proba = model.predict_proba(X_test_scaled)

# Take probability of the positive class (class 1)
proba_pos = proba[:, 1]

# Peek at a few probabilities
print("First 10 probabilities for class 1:")
print(np.round(proba_pos[:10], 3))


First 10 probabilities for class 1:
[0.969 0.    0.559 0.939 0.175 0.    0.003 0.001 0.001 1.   ]


# **Block 8 — Threshold → predictions → confusion matrix + metrics**

This block turns probabilities into final class predictions using a threshold, then evaluates performance. This is where you see the “types of mistakes” via the confusion matrix, and the “summary scores” via accuracy, precision, and recall.

In [8]:
# Choose a threshold (0.50 is the default starting point)
threshold = 0.50

# Convert probabilities into predicted class labels:
# if probability >= threshold -> predict 1
# else -> predict 0
y_pred = (proba_pos >= threshold).astype(int)

# Confusion matrix shows mistake types:
# [[TN, FP],
#  [FN, TP]]
cm = confusion_matrix(y_test, y_pred)

print("Threshold:", threshold)
print("Confusion matrix:\n", cm)

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)  # prevents errors if no predicted positives
rec = recall_score(y_test, y_pred)

print("Accuracy :", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall   :", round(rec, 3))


Threshold: 0.5
Confusion matrix:
 [[52  1]
 [ 1 89]]
Accuracy : 0.986
Precision: 0.989
Recall   : 0.989


# **Block 9 — Show threshold tradeoff**

This block repeats the exact same evaluation at a stricter threshold so you can see the tradeoff clearly with almost no extra code. The point is to make “threshold changes behavior” feel real, not theoretical.

In [9]:
# Try a stricter threshold
threshold2 = 0.80
y_pred2 = (proba_pos >= threshold2).astype(int)

cm2 = confusion_matrix(y_test, y_pred2)

acc2 = accuracy_score(y_test, y_pred2)
prec2 = precision_score(y_test, y_pred2, zero_division=0)
rec2 = recall_score(y_test, y_pred2)

print("\nThreshold:", threshold2)
print("Confusion matrix:\n", cm2)
print("Accuracy :", round(acc2, 3))
print("Precision:", round(prec2, 3))
print("Recall   :", round(rec2, 3))



Threshold: 0.8
Confusion matrix:
 [[52  1]
 [ 6 84]]
Accuracy : 0.951
Precision: 0.988
Recall   : 0.933
